# Joint-span component results

This notebook reads the downloaded ZIP and reports each component's controlled effect. It performs no training, inference, or test-set access.

## Step 1 — Load the artifact


In [1]:
import json,zipfile,sys
from pathlib import Path
from IPython.display import display,Markdown
NAME="joint_span_artifacts.zip"
paths=[Path(NAME),Path("/content")/NAME,Path("/content/drive/MyDrive/CITS4012_A2_joint_span")/NAME]
archive=next((p for p in paths if p.exists()),None)
if archive is None and "google.colab" in sys.modules:
 from google.colab import drive;drive.mount("/content/drive");archive=next((p for p in paths if p.exists()),None)
if archive is None and "google.colab" in sys.modules:
 from google.colab import files
 up=files.upload();assert NAME in up;archive=Path("/content")/NAME;archive.write_bytes(up[NAME])
assert archive is not None
zf=zipfile.ZipFile(archive);summary=json.loads(zf.read("joint_results/joint_summary.json"));print("Loaded",archive,"— no training or inference")


Loaded joint_span_artifacts.zip — no training or inference


## Step 2 — Compare component scores


In [2]:
labels={"ObjectiveControl":"Current objective","QualityWeight":"Quality weighting only","JointSpan":"Joint span only","JointSpan_Quality":"Joint + quality","JointSpan_HardNeg":"Joint + hard negative","JointSpan_All":"Joint + quality + hard negative"}
order=list(labels);ms=("bleu1","meteor","rougeL","em","f1")
lines=["| Model | Params | BLEU-1 | METEOR | ROUGE-L | EM | F1 |","|---|---:|---:|---:|---:|---:|---:|"]
for n in order:
 v=summary["models"][n];lines.append("| "+labels[n]+f" | {v['params']:,} | "+" | ".join(f"{v[m]:.2f}" for m in ms)+" |")
display(Markdown("## One-seed controlled results\n\n"+"\n".join(lines)))


## One-seed controlled results

| Model | Params | BLEU-1 | METEOR | ROUGE-L | EM | F1 |
|---|---:|---:|---:|---:|---:|---:|
| Current objective | 4,094,761 | 38.26 | 31.80 | 43.52 | 27.70 | 43.78 |
| Quality weighting only | 4,094,761 | 39.91 | 34.39 | 45.15 | 28.07 | 45.15 |
| Joint span only | 4,127,544 | 38.27 | 32.11 | 43.18 | 27.88 | 43.43 |
| Joint + quality | 4,127,544 | 38.20 | 31.22 | 43.17 | 27.60 | 43.48 |
| Joint + hard negative | 4,127,544 | 37.79 | 31.72 | 42.81 | 27.23 | 43.06 |
| Joint + quality + hard negative | 4,127,544 | 39.48 | 32.62 | 43.99 | 28.81 | 44.27 |

## Step 3 — Estimate component effects


In [3]:
names={"quality_only":"Quality weighting alone","joint_only":"Joint span scorer alone","quality_given_joint":"Quality weighting added to joint","hard_negative_given_joint":"Hard negative added to joint","hard_negative_given_joint_quality":"Hard negative added to joint + quality","all_vs_control":"All additions versus control"}
lines=["| Controlled contrast | ΔROUGE-L | ΔEM | ΔF1 |","|---|---:|---:|---:|"]
for k,label in names.items():
 v=summary["contrasts"][k];lines.append(f"| {label} | {v['rougeL']:+.2f} | {v['em']:+.2f} | {v['f1']:+.2f} |")
display(Markdown("## Component effects\n\n"+"\n".join(lines)))
best=max(order,key=lambda n:summary["models"][n]["rougeL"]);control=summary["models"]["ObjectiveControl"];b=summary["models"][best]
display(Markdown(f"Best configuration: **{labels[best]}**, ROUGE-L **{b['rougeL']:.2f}** "
                 f"({b['rougeL']-control['rougeL']:+.2f} versus control). Because this is one seed, a promising gain must be confirmed with three seeds."))


## Component effects

| Controlled contrast | ΔROUGE-L | ΔEM | ΔF1 |
|---|---:|---:|---:|
| Quality weighting alone | +1.63 | +0.37 | +1.37 |
| Joint span scorer alone | -0.34 | +0.19 | -0.35 |
| Quality weighting added to joint | -0.01 | -0.28 | +0.04 |
| Hard negative added to joint | -0.37 | -0.65 | -0.37 |
| Hard negative added to joint + quality | +0.82 | +1.21 | +0.79 |
| All additions versus control | +0.47 | +1.12 | +0.49 |

Best configuration: **Quality weighting only**, ROUGE-L **45.15** (+1.63 versus control). Because this is one seed, a promising gain must be confirmed with three seeds.

## Step 4 — Check optimisation histories


In [4]:
# Summarise the saved best epoch and whether later overfitting occurred.
rows=[]
for n in order:
 rid=summary["models"][n]["run_id"];h=json.loads(zf.read(f"joint_results/history/{rid}.json"))
 hist=h["history"];rows.append((labels[n],h["best_epoch"],len(hist),hist[0]["train_loss"],hist[-1]["train_loss"]))
lines=["| Model | Best epoch | Epochs run | First loss | Last loss |","|---|---:|---:|---:|---:|"]
for x in rows:lines.append(f"| {x[0]} | {x[1]} | {x[2]} | {x[3]:.3f} | {x[4]:.3f} |")
display(Markdown("## Optimisation check\n\n"+"\n".join(lines)))


## Optimisation check

| Model | Best epoch | Epochs run | First loss | Last loss |
|---|---:|---:|---:|---:|
| Current objective | 5 | 9 | 5.054 | 2.173 |
| Quality weighting only | 10 | 14 | 4.985 | 1.410 |
| Joint span only | 11 | 15 | 7.209 | 1.878 |
| Joint + quality | 6 | 10 | 7.106 | 2.686 |
| Joint + hard negative | 10 | 14 | 7.831 | 2.286 |
| Joint + quality + hard negative | 7 | 11 | 7.733 | 2.732 |

## Step 5 — Write the conclusion


In [5]:
v=summary["contrasts"];text=f"""# Joint-span objective experiment

In a one-seed controlled comparison, quality weighting alone changed ROUGE-L by
{v['quality_only']['rougeL']:+.2f}, joint span scoring alone by {v['joint_only']['rougeL']:+.2f},
and the complete joint + quality + hard-negative configuration by
{v['all_vs_control']['rougeL']:+.2f} relative to the current objective.

These validation-only results isolate each component under the same split, seed, and frozen
hyperparameters. Any selected configuration should be confirmed over three seeds before use.
The locked test set was not accessed.
"""
Path("joint_span_report.md").write_text(text,encoding="utf-8");display(Markdown("## Report-ready conclusion\n\n"+text));zf.close()


## Report-ready conclusion

# Joint-span objective experiment

In a one-seed controlled comparison, quality weighting alone changed ROUGE-L by
+1.63, joint span scoring alone by -0.34,
and the complete joint + quality + hard-negative configuration by
+0.47 relative to the current objective.

These validation-only results isolate each component under the same split, seed, and frozen
hyperparameters. Any selected configuration should be confirmed over three seeds before use.
The locked test set was not accessed.
